In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

X = pd.read_parquet(
    "../Results/X_final.parquet"
)

y = pd.read_parquet(
    "../Results/y_final.parquet"
)["BinaryLabel"]

print(X.shape)

(2313810, 48)


# Benign training data

In [2]:
X_benign = X[y == 0]

print(
    "Benign training samples:",
    len(X_benign)
)

Benign training samples: 458831


# Isolation Forest

In [3]:
iso = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso.fit(X_benign)

print(
    "Isolation Forest training completed."
)

Isolation Forest training completed.


# Prediction

In [4]:
iso_raw = iso.predict(X)

print("Isolation Forest raw output:")
print(" 1  = Normal")
print("-1  = Anomaly")

print("\nNormal :", np.sum(iso_raw == 1))
print("Anomaly:", np.sum(iso_raw == -1))

Isolation Forest raw output:
 1  = Normal
-1  = Anomaly

Normal : 1984047
Anomaly: 329763


# Convert to binary

In [5]:
y_pred_iso = np.where(
    iso_raw == -1,
    1,
    0
)

print(
    "Binary anomaly prediction completed."
)

Binary anomaly prediction completed.


# Metrics

In [7]:
iso_accuracy = accuracy_score(
    y,
    y_pred_iso
)

iso_precision = precision_score(
    y,
    y_pred_iso,
    zero_division=0
)

iso_recall = recall_score(
    y,
    y_pred_iso,
    zero_division=0
)

iso_f1 = f1_score(
    y,
    y_pred_iso,
    zero_division=0
)

print("Isolation Forest Results")
print("=" * 20)

print(f"Accuracy : {iso_accuracy:.4f}")
print(f"Precision: {iso_precision:.4f}")
print(f"Recall   : {iso_recall:.4f}")
print(f"F1-score : {iso_f1:.4f}")

Isolation Forest Results
Accuracy : 0.3125
Precision: 0.9008
Recall   : 0.1601
F1-score : 0.2719


# Confusion matrix

In [8]:
cm_iso = confusion_matrix(
    y,
    y_pred_iso
)

print(cm_iso)

[[ 426121   32710]
 [1557926  297053]]


# FPR

In [9]:
tn_iso, fp_iso, fn_iso, tp_iso = cm_iso.ravel()

iso_fpr = fp_iso / (
    fp_iso + tn_iso
)

print("TN:", tn_iso)
print("FP:", fp_iso)
print("FN:", fn_iso)
print("TP:", tp_iso)

print(
    f"\nFalse Positive Rate: {iso_fpr:.4f}"
)

TN: 426121
FP: 32710
FN: 1557926
TP: 297053

False Positive Rate: 0.0713


# Report

In [10]:
print(
    classification_report(
        y,
        y_pred_iso,
        target_names=[
            "Benign",
            "Attack"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign       0.21      0.93      0.35    458831
      Attack       0.90      0.16      0.27   1854979

    accuracy                           0.31   2313810
   macro avg       0.56      0.54      0.31   2313810
weighted avg       0.76      0.31      0.29   2313810



# Save

In [11]:
joblib.dump(
    iso,
    "../Models/isolation_forest.joblib"
)

iso_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "FPR"
    ],
    "Value": [
        iso_accuracy,
        iso_precision,
        iso_recall,
        iso_f1,
        iso_fpr
    ]
})

iso_metrics.to_csv(
    "../Results/iso_metrics.csv",
    index=False
)

print("Isolation Forest saved.") 

Isolation Forest saved.
